# Recipe Extraction Notebook

In this notebook, work is done to parse the recipes from the dataset, adding the necessary data to the JSON file. This supplemental data includes:
- Parsed ingredient names (using NLP from `ingredient_parser`)
- Parsed ingredient quantities (using NLP from `ingredient_parser`)
- Parsed ingredient weights (using the parsed ingredient names and quantities, along with a density database to convert to grams)

In [1]:
import json
import sys
from ingredient_parser import parse_ingredient
from ingredient_parser._common import UREG

The Epicurious recipes dataset from Kaggle is used in this project.
> https://www.kaggle.com/datasets/hugodarwood/epirecipes

In [2]:
with open("./epicurious_dataset/full_format_recipes.json", "r") as f:
    recipes = json.load(f)

The dataset, while it contains some nutritional information, is not complete enough. It does not include nutritional information like sugar, saturated fats or fiber. Furthermore, by extracting nutritional information from the ingredient names and quantities, we can also handle recipes that are not in the dataset, which is important should new recipes be generated.

In [3]:
print(json.dumps(recipes[0], indent=4))

{
    "directions": [
        "1. Place the stock, lentils, celery, carrot, thyme, and salt in a medium saucepan and bring to a boil. Reduce heat to low and simmer until the lentils are tender, about 30 minutes, depending on the lentils. (If they begin to dry out, add water as needed.) Remove and discard the thyme. Drain and transfer the mixture to a bowl; let cool.",
        "2. Fold in the tomato, apple, lemon juice, and olive oil. Season with the pepper.",
        "3. To assemble a wrap, place 1 lavash sheet on a clean work surface. Spread some of the lentil mixture on the end nearest you, leaving a 1-inch border. Top with several slices of turkey, then some of the lettuce. Roll up the lavash, slice crosswise, and serve. If using tortillas, spread the lentils in the center, top with the turkey and lettuce, and fold up the bottom, left side, and right side before rolling away from you."
    ],
    "fat": 7.0,
    "date": "2006-09-01T04:00:00.000Z",
    "categories": [
        "Sandwi

It can be seen that the ingredients contain a number of sentence components, such as the preparation method (ex. "diced", "seeded"), quantity (ex. "1 cup", "2 tablespoons"), and the ingredient name itself (ex. "onion", "olive oil"). The goal of this notebook is to parse these components (specifically, the name and quantity) out of the ingredient sentences, and add them to the JSON file for easy use in later stages of the project.

In [4]:
recipes[0]["ingredients"]

['4 cups low-sodium vegetable or chicken stock',
 '1 cup dried brown lentils',
 '1/2 cup dried French green lentils',
 '2 stalks celery, chopped',
 '1 large carrot, peeled and chopped',
 '1 sprig fresh thyme',
 '1 teaspoon kosher salt',
 '1 medium tomato, cored, seeded, and diced',
 '1 small Fuji apple, cored and diced',
 '1 tablespoon freshly squeezed lemon juice',
 '2 teaspoons extra-virgin olive oil',
 'Freshly ground black pepper to taste',
 '3 sheets whole-wheat lavash, cut in half crosswise, or 6 (12-inch) flour tortillas',
 '3/4 pound turkey breast, thinly sliced',
 '1/2 head Bibb lettuce']

In [5]:
sys.path.append("../")  # necessary to import from parent directory

from food_data_extraction import FoodDensityEmbedding

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


The `FoodDensityEmbedding` class from the `food_data_extraction` module is used to convert the parsed ingredient names and quantities into weights in grams, which can be used for nutritional analysis in later stages of the project.

In [6]:
food_density_embedding = FoodDensityEmbedding()

In [7]:
food_density_embedding.load(index_path="../food_data_extraction/food_density_embedding.faiss")

In [8]:
from pint import Unit
from tqdm.auto import tqdm

parsed_recipes = []

for recipe in tqdm(recipes, desc="Parsing Recipes"):
    parsed_recipe = recipe.copy()

    if "ingredients" not in recipe:
        continue

    try:
        parsed_ingredients = [parse_ingredient(ingredient) for ingredient in recipe["ingredients"]]
    except ValueError:
        tqdm.write(f"\033[91m[!] ValueError parsing ingredients for recipe {recipe['title']}\033[0m")
        continue
    except AssertionError:
        tqdm.write(f"\033[91m[!] AssertionError parsing ingredients for recipe {recipe['title']}\033[0m")
        continue
    except IndexError:
        tqdm.write(f"\033[91m[!] IndexError parsing ingredients for recipe {recipe['title']}\033[0m")
        continue

    try:
        # 1. ingredient names
        ingredient_names = [
            ingredient.name[0].text for ingredient in parsed_ingredients
        ]  # TODO only taking the first name, not considering any "Or"
        parsed_recipe["ingredient_names"] = ingredient_names

        # 2. ingredient amounts (consisting of quantity and unit)
        # only taking the first amount (first element). This should not be problematic, since usually, the alternative ingredient will have the same amount
        ingredient_amounts = [ingredient.amount[0] if ingredient.amount else None for ingredient in parsed_ingredients]

        # 2.1 ingredient quantities in terms of each unit
        ingredient_quantities = []
        # 2.2 ingredient weights in grams
        ingredient_weights = []

        for ingredient_name, ingredient_amount in zip(ingredient_names, ingredient_amounts):
            # when no amount is found, the quantity is considered 1, and the default portion is later considered
            if ingredient_amount is None:
                ingredient_quantities.append(
                    1
                )  # TODO investigate, this is a temporary solution to handle ingredients with no amount. Assuming 1 unit of the ingredient, which is not ideal, but better than ignoring the ingredient completely
                ingredient_weights.append(None)
                continue

            density_search = food_density_embedding.search(ingredient_name)
            density = 1 if density_search.empty else density_search["density (g/cm^3)"].iloc[0]
            density = UREG.Quantity(density, "g/cm^3")

            if not hasattr(ingredient_amount, "quantity") or not hasattr(ingredient_amount, "quantity_max"):
                ingredient_quantities.append(None)  # TODO later on, use portion.csv to resolve
                ingredient_weights.append(None)
                continue

            quantity = float(
                (ingredient_amount.quantity + ingredient_amount.quantity_max) / 2
                if ingredient_amount.quantity != "" and ingredient_amount.quantity_max != ""
                else 1
            )
            ingredient_quantities.append(quantity)

            if not isinstance(ingredient_amount.unit, Unit):
                ingredient_weights.append(None)
                continue

            weight = ingredient_amount.convert_to("g", density=density)
            weight = float((weight.quantity + weight.quantity) / 2)
            ingredient_weights.append(weight)

        parsed_recipe["ingredient_quantities"] = ingredient_quantities
        parsed_recipe["ingredient_weights"] = ingredient_weights
    except IndexError:
        tqdm.write(f"\033[91m[!] IndexError parsing ingredients for recipe {recipe['title']}\033[0m")
        continue
    except TypeError:
        tqdm.write(f"\033[91m[!] TypeError parsing ingredients for recipe {recipe['title']}\033[0m")
        continue

    parsed_recipes.append(parsed_recipe)

Parsing Recipes:   0%|          | 0/20130 [00:00<?, ?it/s]

[!] IndexError parsing ingredients for recipe Spicy Noodle Soup 
[!] TypeError parsing ingredients for recipe Asian Pear and Watercress Salad with Sesame Dressing 
[!] TypeError parsing ingredients for recipe Veal Burgers Stuffed with Mozzarella Cheese 
[!] TypeError parsing ingredients for recipe Savoy Cabbage and Arugula Salad 
[!] TypeError parsing ingredients for recipe Cranberry Pear Tart with Gingerbread Crust 
[!] IndexError parsing ingredients for recipe Cold Noodle Salad with Ponzu Sauce 
[!] TypeError parsing ingredients for recipe Sanuki Sea Stock 
[!] TypeError parsing ingredients for recipe Egg Noodle, Chard, and Fontina Torte 
[!] TypeError parsing ingredients for recipe Zucchini Potato Tortilla 
[!] ValueError parsing ingredients for recipe Beet, Carrot, and Apple Juice with Ginger 
[!] IndexError parsing ingredients for recipe Grilled Pork Chops with Tomatillo Salsa 
[!] TypeError parsing ingredients for recipe Cranberry Kumquat Compote 
[!] TypeError parsing ingredient

Finally, the parsed JSON file can be saved to a new file, which will be used in later stages of the project.

In [10]:
with open("./parsed_recipes.json", "w") as f:
    json.dump(parsed_recipes, f, indent=4, default=str)